# 🧠 Esercitazione 4 — Costruire una rete neurale con PyTorch

**new ALF – Tutorato** · Corso di Laurea in Informatica · AA 2026/27

Oggi costruiamo una rete neurale **pezzo per pezzo** con **PyTorch**, la libreria più usata per il deep learning, e guardiamo cosa succede al suo interno: come è fatto un layer, cosa calcola, come cambia l'ultimo layer tra regressione e classificazione, come si aggiornano i pesi durante l'addestramento e cosa fare quando la rete impara a memoria.

Per concentrarci sulla rete usiamo **dati sintetici**: numeri generati da una formula nota, più un po' di rumore. Il compito che seguiamo dall'inizio alla fine è una **regressione**; con gli stessi dati costruiamo anche una classificazione binaria e una a tre classi, che compaiono nella Parte 3 (l'ultimo layer) e negli approfondimenti.

> ### ⚠️ Prima di tutto: fai la tua copia
> Questo notebook è condiviso in **sola lettura**: puoi eseguirlo e modificarlo, ma le modifiche **non verranno salvate**.
> Vai su **File → Salva una copia in Drive**. Si aprirà una nuova scheda con la *tua* copia (la trovi poi su Google Drive, nella cartella *Colab Notebooks*): lavora lì.

### Come si usa questo notebook
- Ogni **cella** si esegue con **Shift + Invio** (o con il tasto ▶ a sinistra). Esegui le celle **in ordine**, dall'alto verso il basso.
- Le celle segnate con **✏️** sono esercizi: modifica il codice dove indicato. Quando sotto c'è un blocco **💡 Soluzione** chiuso, prova prima da solo. Le celle successive funzionano anche se lasci l'esercizio com'è, **tranne nella 4.6**: lì, se non riesci, copia la soluzione prima di andare avanti.
- I box **🐍** spiegano le cose nuove di Python e di PyTorch che incontriamo strada facendo.
- Le sezioni segnate con **⏩** sono approfondimenti.
- I box **📋** dicono quale problema stiamo risolvendo: i dati, il target `y`, le feature `X` e il modo di valutare il modello.
- I box **🤔** sono domande su cui riflettere (spesso con la risposta chiusa sotto), i **📌** i concetti chiave.
- Se qualcosa si rompe: menu **Runtime → Riavvia sessione ed esegui tutto** (*Restart session and run all*).

In [ ]:
#@title ⚙️ Setup: le librerie (esegui la cella)
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

print("✅ PyTorch", torch.__version__)

In [ ]:
#@title 🛠️ Funzioni di supporto per i grafici (esegui la cella, non serve leggerla)
def disegna_curve(storie, titolo="", max_loss=None):
    """Curve della loss per epoca. `storie` è un dizionario {nome: tabella restituita da addestra}.
    Linea tratteggiata = training, linea continua = validazione, stesso colore per la stessa rete."""
    fig, ax = plt.subplots(figsize=(8, 4))
    for i, (nome, storia) in enumerate(storie.items()):
        colore = f"C{i}"
        ax.plot(storia["epoca"], storia["loss_train"], "--", color=colore, label=f"{nome} · training")
        ax.plot(storia["epoca"], storia["loss_val"], "-", color=colore, label=f"{nome} · validazione")
    ax.set_xlabel("epoca"); ax.set_ylabel("loss"); ax.set_title(titolo)
    if max_loss is not None:
        ax.set_ylim(0, max_loss)
    ax.grid(alpha=0.3); ax.legend(fontsize=8)
    plt.show()

def confronta_previsioni(veri, previsioni, titoli):
    """Valore vero contro valore stimato, un grafico per modello: più i punti stanno sulla diagonale, meglio è."""
    fig, assi = plt.subplots(1, len(previsioni), figsize=(5 * len(previsioni), 4.2))
    for ax, prev, titolo in zip(np.atleast_1d(assi), previsioni, titoli):
        ax.scatter(veri, prev, s=10, alpha=0.5)
        estremi = [min(veri.min(), prev.min()), max(veri.max(), prev.max())]
        ax.plot(estremi, estremi, "k--", linewidth=1)
        ax.set_xlabel("y vero"); ax.set_ylabel("y stimato"); ax.set_title(titolo)
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

print("✅ Funzioni pronte: disegna_curve, confronta_previsioni")

---
# Parte 1 — I dati e i tensori

### 1.1 Dati sintetici, tre compiti
Generiamo 2000 esempi con `make_friedman1` di scikit-learn. Ogni esempio ha **10 feature**, numeri a caso tra 0 e 1, e un valore $y$ che dipende da alcune di queste feature in modo **non lineare**, più un po' di rumore. La formula non ci interessa: ci basta sapere che una retta non basta a descriverla.

Da $y$ ricaviamo tre target diversi:

> 📋 **Tre problemi, stesse feature**
> - **Feature `X`:** le 10 colonne da `x1` a `x10`
> - **`y_reg`**: il valore $y$ → **regressione**, valutata con l'MSE. È il compito che seguiamo in tutta l'esercitazione
> - **`y_bin`**: 1 se $y$ è sopra la mediana, 0 altrimenti → **classificazione binaria**, valutata con l'accuracy (Parte 3 e approfondimenti ⏩)
> - **`y_cls`**: 0, 1 o 2 se $y$ è nel terzo più basso, in quello centrale o in quello più alto → **classificazione a tre classi**, valutata con l'accuracy (Parte 3 e approfondimenti ⏩)

In [ ]:
from sklearn.datasets import make_friedman1

X, y = make_friedman1(n_samples=2000, n_features=10, noise=1.0, random_state=0)

feature = [f"x{i}" for i in range(1, 11)]          # i nomi delle colonne: x1, x2, ..., x10
dati = pd.DataFrame(X, columns=feature)
dati["y_reg"] = y
dati["y_bin"] = (y > np.median(y)).astype(int)     # sopra la mediana → 1
dati["y_cls"] = pd.qcut(y, 3, labels=False)        # qcut: tre fasce con lo stesso numero di esempi → 0, 1, 2

dati.head().round(2)

> 🐍 `[f"x{i}" for i in range(1, 11)]` è una *list comprehension*: costruisce la lista `["x1", "x2", ..., "x10"]` ripetendo l'espressione per ogni `i`. `pd.qcut(y, 3, labels=False)` divide i valori in 3 fasce con lo stesso numero di esempi e restituisce il numero della fascia.

### 1.2 Training, validation e test set, e la standardizzazione
Dividiamo gli esempi: **60%** per addestrare, **20%** per confrontare le reti (validazione), **20%** di test.

Poi **standardizziamo** le feature: a ognuna togliamo la media e la dividiamo per la deviazione standard, così hanno tutte media 0 e deviazione standard 1. Le reti imparano molto meglio con numeri piccoli e centrati sullo zero. Media e deviazione standard si calcolano **solo sul training set** e poi si applicano anche a validazione e test: se le calcolassimo su tutti i dati, la rete riceverebbe informazioni sul test set (*data leakage*).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

train, resto = train_test_split(dati, test_size=0.4, random_state=0)    # 60% training
val, test = train_test_split(resto, test_size=0.5, random_state=0)      # 20% validazione, 20% test

scaler = StandardScaler().fit(train[feature])      # media e deviazione standard SOLO dal training
print("Esempi:", len(train), "training,", len(val), "validazione,", len(test), "test")
print("Media delle prime 3 feature dopo la standardizzazione:", scaler.transform(train[feature]).mean(axis=0)[:3].round(3))

### 1.3 Cosa sono i tensori
PyTorch lavora con i **tensori**. Un tensore è una tabella di numeri con un numero qualsiasi di **dimensioni** (o *assi*):

| Dimensioni | Cos'è | Forma (`shape`) | Esempio |
|---|---|---|---|
| 0 | un numero (*scalare*) | `()` | una loss |
| 1 | un **vettore** | `(n,)` | le 10 feature di un esempio, i bias di un layer |
| 2 | una **matrice**, cioè una tabella | `(righe, colonne)` | i nostri dati: una riga per esempio, una colonna per feature; i pesi di un layer |
| 3 o più | tabelle di tabelle | `(a, b, c, ...)` | un'immagine a colori (3 × altezza × larghezza), un batch di immagini |

Un tensore assomiglia a un array di NumPy, con due cose in più che servono alle reti: può stare sulla **GPU**, dove i calcoli tra matrici sono molto più veloci, e PyTorch può **tenere traccia dei calcoli** fatti con lui per calcolarne i gradienti (lo vedremo nella Parte 4). Ogni tensore ha:
- una **forma** (`.shape`): quanti elementi ci sono lungo ogni dimensione;
- un **tipo** (`.dtype`): per esempio `float32` (numeri con la virgola a 32 bit, il tipo standard delle reti) o `int64` (interi, in PyTorch `torch.long`);
- un **dispositivo** (`.device`): `cpu` o `cuda` (la GPU).

In [ ]:
scalare = torch.tensor(3.5)
vettore = torch.tensor([1.0, 2.0, 3.0])
matrice = torch.tensor([[1.0, 2.0, 3.0],
                        [4.0, 5.0, 6.0]])
cubo = torch.zeros(2, 3, 4)                     # 2 tabelle da 3 righe e 4 colonne, piene di zeri

for nome, t in [("scalare", scalare), ("vettore", vettore), ("matrice", matrice), ("cubo", cubo)]:
    print(f"{nome:8s} dimensioni: {t.ndim}   forma: {str(tuple(t.shape)):10s} tipo: {t.dtype}   dispositivo: {t.device}")

print("\nCon numeri interi il tipo cambia:", torch.tensor([1, 2, 3]).dtype)

Le operazioni sui tensori si fanno **su tutti gli elementi insieme**, senza `for`. Vediamo una alla volta quelle che useremo, sui tensori `matrice` e `vettore` della cella precedente. Ogni cella stampa anche le **forme**: quella generale, con le lettere, e quella dell'esempio. Chiamiamo $n$ il numero di righe e $m$ il numero di colonne della matrice: qui $n = 2$ e $m = 3$, e il vettore ha $m = 3$ elementi.

**Moltiplicare per un numero**: ogni elemento viene moltiplicato.

In [ ]:
print("matrice:\n", matrice)
print("\nmatrice * 10:\n", matrice * 10)
print(f"\nForme: (n, m) * numero → (n, m)      qui: {tuple(matrice.shape)} * 10 → {tuple((matrice * 10).shape)}")

**Sommare un vettore a una matrice**: il vettore si somma a **ogni riga**. PyTorch lo "ripete" da solo per tutte le righe (si chiama *broadcasting*). È quello che succede con il bias di un layer.

In [ ]:
print("matrice:\n", matrice)
print("\nvettore:", vettore)
print("\nmatrice + vettore:\n", matrice + vettore)
print(f"\nForme: (n, m) + (m,) → (n, m)      qui: {tuple(matrice.shape)} + {tuple(vettore.shape)} → {tuple((matrice + vettore).shape)}")

**Prodotto tra matrici `@`**: ogni riga della matrice viene moltiplicata per il vettore elemento per elemento, e i prodotti si sommano:

$$1 \cdot 1 + 2 \cdot 2 + 3 \cdot 3 = 14 \qquad\qquad 4 \cdot 1 + 5 \cdot 2 + 6 \cdot 3 = 32$$

È la **somma pesata** di un neurone, fatta per ogni riga in un colpo solo.

In [ ]:
print("matrice:\n", matrice)
print("\nvettore:", vettore)
print("\nmatrice @ vettore:", matrice @ vettore)
print(f"\nForme: (n, m) @ (m,) → (n,)      qui: {tuple(matrice.shape)} @ {tuple(vettore.shape)} → {tuple((matrice @ vettore).shape)}")

**Trasposta `.T`**: righe e colonne si scambiano, e la forma (2, 3) diventa (3, 2).

In [ ]:
print("matrice, forma", tuple(matrice.shape), ":\n", matrice)
print("\nmatrice.T, forma", tuple(matrice.T.shape), ":\n", matrice.T)
print(f"\nForme: (n, m).T → (m, n)      qui: {tuple(matrice.shape)}.T → {tuple(matrice.T.shape)}")

**Prendere righe e colonne**: tra parentesi quadre si dice cosa prendere lungo ogni dimensione, e `:` vuol dire "tutto". `matrice[:, 1]` è "tutte le righe, colonna 1" (si conta da 0).

Attenzione: `matrice[0]` e `matrice[:1]` contengono gli stessi numeri, ma il primo è un **vettore** e il secondo una **tabella con una riga**. Per le reti la differenza conta: un esempio da solo è un vettore, mentre un **batch** di esempi è una tabella con un esempio per riga (lo vediamo nella 2.2).

In [ ]:
print("matrice:\n", matrice)
print("\nmatrice[0]:   ", matrice[0])
print("matrice[:1]:  ", matrice[:1])
print("matrice[:, 1]:", matrice[:, 1])
print(f"\nForme: matrice[0]    (n, m) → (m,)      qui: {tuple(matrice.shape)} → {tuple(matrice[0].shape)}")
print(f"       matrice[:1]   (n, m) → (1, m)    qui: {tuple(matrice.shape)} → {tuple(matrice[:1].shape)}")
print(f"       matrice[:, 1] (n, m) → (n,)      qui: {tuple(matrice.shape)} → {tuple(matrice[:, 1].shape)}")

**Medie** (e allo stesso modo somme, minimi, massimi): su tutto il tensore, oppure lungo una dimensione. `dim=0` scorre lungo le righe e dà la media di ogni **colonna**; `dim=1` scorre lungo le colonne e dà la media di ogni **riga**.

In [ ]:
print("matrice:\n", matrice)
print("\nmatrice.mean():     ", matrice.mean())
print("matrice.mean(dim=0):", matrice.mean(dim=0))
print("matrice.mean(dim=1):", matrice.mean(dim=1))
print(f"\nForme: .mean()       (n, m) → ()        qui: {tuple(matrice.shape)} → {tuple(matrice.mean().shape)}")
print(f"       .mean(dim=0)  (n, m) → (m,)      qui: {tuple(matrice.shape)} → {tuple(matrice.mean(dim=0).shape)}")
print(f"       .mean(dim=1)  (n, m) → (n,)      qui: {tuple(matrice.shape)} → {tuple(matrice.mean(dim=1).shape)}")

**Da NumPy a tensore e ritorno**: `torch.tensor(array)` crea un tensore da un array NumPy, `.numpy()` fa il contrario. Ci servirà per passare i dati da pandas e scikit-learn a PyTorch, e i risultati da PyTorch a matplotlib.

In [ ]:
array = np.array([[1.5, 2.5], [3.5, 4.5]])
tensore = torch.tensor(array, dtype=torch.float32)
print("Array NumPy:\n", array)
print("\nTensore:\n", tensore)
print("\nDi nuovo NumPy:\n", tensore.numpy())
print(f"\nForme: la forma non cambia, (n, m) → (n, m)      qui: {array.shape} → {tuple(tensore.shape)} → {tensore.numpy().shape}")

### 1.4 I nostri dati come tensori
Ora trasformiamo le tabelle di pandas in tensori.

Ogni loss vuole il target in una forma precisa, e lo vedremo nella Parte 3:
- **regressione** e **classificazione binaria**: una colonna di numeri con la virgola, forma `(n, 1)`, come l'uscita della rete;
- **tre classi**: un vettore di **numeri interi** (0, 1, 2), forma `(n,)`.

> 🐍 `torch.tensor(..., dtype=torch.float32)` crea un tensore di numeri con la virgola a 32 bit, il tipo standard delle reti. `torch.long` sono gli interi. `.reshape(-1, 1)` trasforma un vettore di `n` numeri in una tabella `n × 1` (il `-1` vuol dire "quante righe servono").

In [ ]:
def in_tensori(tabella):
    """Feature standardizzate e i tre target di una tabella, come tensori PyTorch."""
    X = torch.tensor(scaler.transform(tabella[feature]), dtype=torch.float32)
    y_reg = torch.tensor(tabella["y_reg"].to_numpy(), dtype=torch.float32).reshape(-1, 1)
    y_bin = torch.tensor(tabella["y_bin"].to_numpy(), dtype=torch.float32).reshape(-1, 1)
    y_cls = torch.tensor(tabella["y_cls"].to_numpy(), dtype=torch.long)
    return X, y_reg, y_bin, y_cls

X_train, yreg_train, ybin_train, ycls_train = in_tensori(train)
X_val, yreg_val, ybin_val, ycls_val = in_tensori(val)
X_test, yreg_test, ybin_test, ycls_test = in_tensori(test)

print("X_train:   ", tuple(X_train.shape), X_train.dtype)
print("yreg_train:", tuple(yreg_train.shape), yreg_train.dtype)
print("ybin_train:", tuple(ybin_train.shape), ybin_train.dtype)
print("ycls_train:", tuple(ycls_train.shape), ycls_train.dtype)

---
# Parte 2 — Costruire la rete

### 2.1 Il percettrone: un neurone
Il mattone di ogni rete è il **neurone artificiale**, o **percettrone** (Rosenblatt, 1958). Riceve degli ingressi $x_1, \dots, x_n$, ognuno con il suo **peso** $w_i$, e fa due cose:

1. calcola la **somma pesata** degli ingressi più un **bias** $b$: $\quad z = w_1 x_1 + w_2 x_2 + \dots + w_n x_n + b = w \cdot x + b$
2. passa $z$ in una **funzione di attivazione** $f$: $\quad a = f(z)$

Il peso dice quanto conta ogni ingresso (e con che segno), il bias sposta la soglia oltre la quale il neurone "si accende". Il percettrone originale usava un **gradino**: 1 se $z > 0$, 0 altrimenti. Oggi si usano funzioni più morbide, che vediamo nella 2.3.

Rifacciamo con i tensori l'esempio della lezione: 3 ingressi, 3 pesi e un bias.

In [ ]:
x = torch.tensor([2.0, -1.0, 3.0])      # gli ingressi
w = torch.tensor([0.5, 1.0, -0.2])      # un peso per ogni ingresso
b = torch.tensor(1.0)                   # il bias

print("w * x (prodotti, elemento per elemento):", w * x)
z = (w * x).sum() + b                   # somma pesata + bias
print(f"z = somma dei prodotti + b = {z.item():.3f}")
print(f"lo stesso con @:  w @ x + b = {(w @ x + b).item():.3f}")

print("\nL'uscita a = f(z) con diverse funzioni di attivazione:")
print(f"  gradino (percettrone originale): {(z > 0).float().item():.3f}")
print(f"  ReLU:     {torch.relu(z).item():.3f}")
print(f"  sigmoide: {torch.sigmoid(z).item():.3f}")
print(f"  tanh:     {torch.tanh(z).item():.3f}")

### 2.2 Tanti neuroni insieme: `nn.Linear`
Un **layer denso** (*fully connected*) è un gruppo di percettroni affiancati che ricevono **tutti gli stessi ingressi**: ognuno fa la sua somma pesata, con i suoi pesi e il suo bias, e restituisce un numero. In PyTorch è `nn.Linear(n_ingressi, n_uscite)`, dove `n_uscite` è il numero di neuroni.

**Un esempio alla volta.** Riprendiamo l'esempio della 2.1, con 3 ingressi $x = (2, -1, 3)$, e mettiamo **2 neuroni**: il primo è il percettrone della 2.1, il secondo ha pesi e bias diversi. PyTorch tiene i pesi di un layer in una tabella $W$ con **una riga per neurone** e una colonna per ingresso:

$$W = \begin{bmatrix} 0.5 & 1 & -0.2 \\ 1 & 0 & 1 \end{bmatrix} \quad \text{forma } (2, 3) \qquad\qquad b = (1, -2) \quad \text{forma } (2,)$$

Con un esempio solo, le due somme pesate si possono calcolare con un prodotto `@` in **due modi equivalenti**:
- `W @ x`: ogni riga di $W$, cioè ogni neurone, fa il prodotto con $x$: (2, 3) @ (3,) → (2,);
- `x @ W.T`: l'esempio sta a **sinistra**, e allora i pesi vanno **trasposti**, perché ogni neurone deve stare in una colonna: (3,) @ (3, 2) → (2,).

PyTorch usa sempre la seconda forma, perché funziona allo stesso modo anche quando gli esempi sono tanti (lo vediamo alla fine di questa sezione). Il layer calcola quindi

$$x\, W^T + b \qquad\qquad (3,)\; @\; (3, 2) \;+\; (2,) \;\to\; (2,)$$

Il primo numero del risultato è la somma pesata del primo neurone, il secondo quella del secondo neurone:

$$2 \cdot 0.5 + (-1) \cdot 1 + 3 \cdot (-0.2) + 1 = 0.4 \qquad\qquad 2 \cdot 1 + (-1) \cdot 0 + 3 \cdot 1 - 2 = 3$$

In [ ]:
x = torch.tensor([2.0, -1.0, 3.0])          # un esempio con 3 ingressi
W = torch.tensor([[0.5, 1.0, -0.2],         # pesi del neurone 1 (il percettrone della 2.1)
                  [1.0, 0.0, 1.0]])         # pesi del neurone 2
b = torch.tensor([1.0, -2.0])               # un bias per neurone

print("x:", x, "  forma", tuple(x.shape))
print("\nW, forma", tuple(W.shape), ":\n", W)
print("\nW.T, forma", tuple(W.T.shape), ":\n", W.T)
print("\nb:", b, "  forma", tuple(b.shape))

print("\nW @ x + b:  ", W @ x + b, f"  forme: {tuple(W.shape)} @ {tuple(x.shape)} + {tuple(b.shape)} → {tuple((W @ x + b).shape)}")
uscita = x @ W.T + b
print("x @ W.T + b:", uscita, f"  forme: {tuple(x.shape)} @ {tuple(W.T.shape)} + {tuple(b.shape)} → {tuple(uscita.shape)}")

Lo stesso con `nn.Linear`: creiamo un layer con 3 ingressi e 2 neuroni e gli scriviamo dentro i pesi e i bias di prima. Di solito non si fa (i pesi li trova l'addestramento), ma così vediamo che `strato.weight` è proprio la nostra $W$, una riga per neurone.

> 🐍 `with torch.no_grad():` dice a PyTorch di non tenere traccia dei calcoli fatti nel blocco (qui stiamo solo scrivendo dei numeri). `.copy_(W)` copia i valori di `W` dentro il tensore dei pesi.

In [ ]:
strato = nn.Linear(3, 2)                    # 3 ingressi → 2 neuroni
with torch.no_grad():
    strato.weight.copy_(W)
    strato.bias.copy_(b)

print("strato.weight, forma", tuple(strato.weight.shape), ":\n", strato.weight)
print("\nstrato(x):", strato(x))

Stesso risultato. Il `grad_fn=...` che compare accanto ai numeri è il segno che PyTorch **tiene traccia dei calcoli**: gli servirà per calcolare i gradienti nel backward (Parte 4).

**Un layer sui nostri dati.** Ora un layer con 10 ingressi (le nostre feature) e 4 neuroni. Questa volta i pesi sono quelli **casuali** con cui PyTorch crea ogni layer: con `torch.manual_seed` fissiamo il seme, così otteniamo sempre gli stessi.

In [ ]:
torch.manual_seed(0)
strato = nn.Linear(10, 4)                   # 10 ingressi → 4 neuroni

x0 = X_train[0]                             # il primo esempio di training: 10 numeri
print("x0:", x0, "  forma", tuple(x0.shape))
print("\nPesi W:", tuple(strato.weight.shape), "→ una riga per neurone, una colonna per ingresso")
print("Bias b:", tuple(strato.bias.shape))

print("\nstrato(x0):", strato(x0))
print(f"Forme: {tuple(x0.shape)} @ {tuple(strato.weight.T.shape)} + {tuple(strato.bias.shape)} → {tuple(strato(x0).shape)}")

### ✏️ Esercizio: il calcolo del layer, scritto da te
Calcola l'uscita del layer per `x0` **senza chiamarlo**, con la formula $x\, W^T + b$. Ti servono `strato.weight` e `strato.bias`, il prodotto `@` e la trasposta `.T`. Se il risultato coincide con quello di `strato(x0)`, la cella stampa ✅.

In [ ]:
# ✏️ sostituisci torch.zeros(4) con la formula
mia_uscita = torch.zeros(4)

print("Il tuo calcolo:", mia_uscita)
print("nn.Linear:     ", strato(x0))
print("✅ Coincidono!" if torch.allclose(mia_uscita, strato(x0)) else "❌ Non ancora: riprova")

<details>
<summary><b>💡 Soluzione</b> (clicca per aprire, ma prima prova da solo!)</summary>

```python
mia_uscita = x0 @ strato.weight.T + strato.bias
```

`x0` ha forma (10,), `strato.weight.T` ha forma (10, 4): il prodotto ha forma (4,), una somma pesata per ogni neurone, e a ognuna si somma il suo bias.

</details>

**Un batch di esempi.** Durante l'addestramento non passiamo alla rete un esempio alla volta, ma un **batch**: una tabella $X$ con un esempio per riga, di forma $(n, 10)$. La formula è la stessa, con $X$ al posto di $x$:

$$X\, W^T + b \qquad\qquad (n, 10)\; @\; (10, 4) \;+\; (4,) \;\to\; (n, 4)$$

Ecco perché PyTorch usa la forma con la trasposta: con un batch, `W @ X` non si può più fare, perché (4, 10) @ ($n$, 10) non torna (le 10 colonne di $W$ non corrispondono alle $n$ righe di $X$). Con $X$ a sinistra e $W^T$ a destra invece sì, e il risultato ha un esempio per riga e un neurone per colonna. **Ogni riga** di $X$ viene moltiplicata per la **stessa** $W^T$, perché i pesi appartengono al layer e non all'esempio, e a ogni riga si somma lo stesso $b$ (il *broadcasting* della 1.3). È lo stesso calcolo di prima, ripetuto per tutti gli esempi del batch con una sola moltiplicazione tra matrici: è per questo che le GPU, velocissime a moltiplicare matrici, rendono veloci le reti.

In [ ]:
batch = X_train[:5]                         # i primi 5 esempi: una tabella 5 × 10
uscite = strato(batch)

print("batch: forma", tuple(batch.shape))
print("\nstrato(batch):\n", uscite)
print(f"Forme: {tuple(batch.shape)} @ {tuple(strato.weight.T.shape)} + {tuple(strato.bias.shape)} → {tuple(uscite.shape)}")
print("\nLa prima riga è uguale a strato(x0)?", torch.allclose(uscite[0], strato(x0)))

### 2.3 Le funzioni di attivazione
Dopo la somma pesata, ogni neurone nascosto passa il risultato $z$ in una **funzione di attivazione** non lineare. Ecco le più usate, ognuna con la sua **derivata** (tratteggiata), che è quella che conta durante l'addestramento: dove la derivata è quasi zero la funzione è piatta (*satura*) e il gradiente quasi non passa.

| Funzione | Formula | Valori | Dove si usa |
|---|---|---|---|
| Sigmoide | $\dfrac{1}{1 + e^{-z}}$ | tra 0 e 1 | uscita della classificazione binaria (probabilità) |
| tanh | $\dfrac{e^z - e^{-z}}{e^z + e^{-z}}$ | tra −1 e 1, centrata in 0 | layer nascosti (reti più vecchie, reti per sequenze) |
| ReLU | $\max(0, z)$ | da 0 in su | layer nascosti: la scelta standard |
| Leaky ReLU | $\max(\alpha z, z)$ | tutti i numeri | come la ReLU, ma i negativi non vengono azzerati del tutto |
| GELU | $z \cdot \Phi(z)$ | quasi tutti i numeri | una ReLU "smussata", usata nei transformer |

> 🐍 Le derivate le calcola PyTorch. Con `requires_grad=True` PyTorch tiene traccia dei calcoli fatti a partire da `z`, e `torch.autograd.grad(uscita.sum(), z)` restituisce la derivata dell'uscita in ogni punto: è lo stesso meccanismo (l'*autograd*) che userà l'addestramento nella Parte 4. `.detach()` stacca un tensore da questo tracciamento, come serve a matplotlib per disegnarlo.

In [ ]:
z = torch.linspace(-3, 3, 301, requires_grad=True)       # 301 punti tra -3 e 3, di cui vogliamo anche le derivate

attivazioni = {
    "Sigmoide": nn.Sigmoid(),
    "tanh": nn.Tanh(),
    "ReLU": nn.ReLU(),
    "Leaky ReLU (α = 0.1)": nn.LeakyReLU(0.1),           # α di solito è 0.01: qui 0.1 per vederlo nel grafico
    "GELU": nn.GELU(),
}

fig, assi = plt.subplots(1, len(attivazioni), figsize=(18, 3.6), sharey=True)
for ax, (nome, funzione) in zip(assi, attivazioni.items()):
    uscita = funzione(z)
    derivata = torch.autograd.grad(uscita.sum(), z)[0]   # la derivata in ognuno dei 301 punti
    ax.plot(z.detach(), uscita.detach(), linewidth=2.5, label="f(z)")
    ax.plot(z.detach(), derivata, "--", linewidth=1.5, label="derivata f'(z)")
    ax.axhline(0, color="k", linewidth=0.8); ax.axvline(0, color="k", linewidth=0.8)
    ax.set_title(nome); ax.set_xlabel("z"); ax.set_ylim(-1.3, 3.1); ax.grid(alpha=0.3)
assi[0].legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

- **Sigmoide** e **tanh** si appiattiscono ai lati: per $z$ lontano da 0 la derivata è quasi zero. La derivata della sigmoide non supera mai $0.25$.
- La **ReLU** azzera i negativi e lascia passare i positivi così come sono: la sua derivata vale 0 a sinistra e **1** a destra.
- **Leaky ReLU** e **GELU** sono varianti della ReLU che non azzerano del tutto i valori negativi.

Nelle nostre reti useremo la ReLU nei layer nascosti; sigmoide e softmax torneranno nella Parte 3, sull'ultimo layer.

### 2.4 Più layer in fila: `nn.Sequential`

Una rete si costruisce mettendo i layer **in fila**: l'uscita di ognuno è l'ingresso del successivo. Tra un layer lineare e l'altro serve una **funzione di attivazione** non lineare, qui la **ReLU**: senza, due layer lineari in fila sarebbero ancora una sola trasformazione lineare.

Costruiamo una rete con 10 ingressi, due layer nascosti da 32 e 16 neuroni e 1 uscita:

$$10 \;\xrightarrow{\text{Linear + ReLU}}\; 32 \;\xrightarrow{\text{Linear + ReLU}}\; 16 \;\xrightarrow{\text{Linear}}\; 1$$

In [ ]:
torch.manual_seed(0)
rete = nn.Sequential(
    nn.Linear(10, 32), nn.ReLU(),
    nn.Linear(32, 16), nn.ReLU(),
    nn.Linear(16, 1),
)
print(rete)

print("\nI parametri, layer per layer:")
for nome, parametro in rete.named_parameters():          # nome e tensore di ogni gruppo di parametri
    print(f"  {nome:9s} forma {str(tuple(parametro.shape)):9s} → {parametro.numel()} numeri")
print("Totale:", sum(p.numel() for p in rete.parameters()))   # .numel(): quanti numeri ci sono in un tensore

Ogni layer ha $(n_{in} \times n_{out}) + n_{out}$ parametri: $10 \cdot 32 + 32 = 352$, $32 \cdot 16 + 16 = 528$, $16 \cdot 1 + 1 = 17$, in totale **897**. I nomi `0.weight`, `2.weight`, `4.weight` dicono la posizione del layer nella fila: le posizioni 1 e 3 sono le ReLU, che non hanno parametri.

### 2.5 La rete come classe: un `MLP` con `nn.Module`
`nn.Sequential` va bene per le reti semplici. Il modo standard di scrivere una rete in PyTorch è una **classe** che eredita da `nn.Module`, con due parti:
- `__init__`: crea i layer, una volta sola;
- `forward`: dice cosa succede quando un dato attraversa la rete.

Una rete fatta così, con layer densi in fila e una non linearità tra uno e l'altro, si chiama **percettrone multistrato**, in inglese *MultiLayer Perceptron* o **MLP**: chiamiamo così anche la nostra classe. `MLP` costruisce i layer nascosti a partire da una **lista** di numeri di neuroni: `[32, 16]` vuol dire due layer nascosti da 32 e 16 neuroni. Così, per provare architetture diverse, basta cambiare la lista. Il parametro `dropout` lo useremo nella Parte 5.

> 🐍 Una **classe** è lo stampo per creare oggetti. `__init__` viene eseguito quando l'oggetto nasce (`MLP(...)`), e `self` è l'oggetto stesso: `self.strati = ...` gli attacca un attributo. `class MLP(nn.Module)` vuol dire che `MLP` **eredita** tutto quello che sa fare un `nn.Module` di PyTorch (per esempio `.parameters()`), e `super().__init__()` prepara quella parte. Quando scriviamo `rete(x)`, PyTorch chiama `forward(x)`.

In [ ]:
class MLP(nn.Module):
    def __init__(self, n_ingressi, neuroni_nascosti, n_uscite, dropout=0.0):
        super().__init__()
        strati = []
        n_prima = n_ingressi                       # quanti numeri entrano nel prossimo layer
        for n in neuroni_nascosti:                 # un layer nascosto per ogni numero della lista
            strati.append(nn.Linear(n_prima, n))
            strati.append(nn.ReLU())
            if dropout > 0:
                strati.append(nn.Dropout(dropout))    # (Parte 5)
            n_prima = n
        strati.append(nn.Linear(n_prima, n_uscite))   # layer di uscita, senza attivazione
        self.strati = nn.Sequential(*strati)          # 🐍 *strati "spacchetta" la lista

    def forward(self, x):
        return self.strati(x)

torch.manual_seed(0)
rete = MLP(n_ingressi=10, neuroni_nascosti=[32, 16], n_uscite=1)
print(rete)
print("Parametri:", sum(p.numel() for p in rete.parameters()))
print("\nUscita per 5 esempi:", tuple(rete(X_train[:5]).shape))

È la stessa rete della 2.4, con gli stessi 897 parametri. Nota le forme: 5 esempi entrano **insieme**, come una tabella 5 × 10 (un *batch*), ed escono come una tabella 5 × 1, una stima per esempio.

### 2.6 Dentro la rete
Facciamo attraversare la rete a **un solo esempio**, un layer alla volta, e guardiamo cosa esce da ognuno.

In [ ]:
h = X_train[:1]                                    # il primo esempio di training
with torch.no_grad():                              # stiamo solo guardando: niente gradienti
    for strato in rete.strati:                     # 🐍 un nn.Sequential si può scorrere con un for
        h = strato(h)
        nome = strato.__class__.__name__           # "Linear" o "ReLU"
        print(f"{nome:6s} → forma {str(tuple(h.shape)):8s} zeri: {(h == 0).sum().item():2d}   "
              f"primi valori: {h[0, :5].numpy().round(2)}")

with torch.no_grad():                              # il primo layer nascosto, prima e dopo la ReLU
    z1 = rete.strati[0](X_train[:1])[0]
    h1 = rete.strati[1](z1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3), sharey=True)
ax1.bar(range(32), z1.numpy(), color=["C3" if v < 0 else "C0" for v in z1])
ax1.set_title("Primo layer nascosto: dopo Linear"); ax1.set_xlabel("neurone")
ax2.bar(range(32), h1.numpy(), color="C0")
ax2.set_title("... e dopo la ReLU"); ax2.set_xlabel("neurone")
for ax in (ax1, ax2):
    ax.axhline(0, color="k", linewidth=0.8); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

L'esempio entra con 10 numeri, diventa 32 numeri nel primo layer nascosto, 16 nel secondo e 1 in uscita. A sinistra le 32 somme pesate del primo layer: le barre rosse sono negative. La ReLU le **azzera** e lascia passare le positive. Questi vettori intermedi sono la **rappresentazione** che la rete costruisce dell'esempio: ogni layer trasforma quella del layer precedente. Per ora i pesi sono casuali e i numeri non significano niente: con l'addestramento diventeranno utili al compito.

---
# Parte 3 — L'ultimo layer: regressione o classificazione?

Fino all'ultimo layer, una rete per la regressione e una per la classificazione **possono essere identiche**. Cambiano solo il numero di uscite e il modo di leggerle. I numeri che escono dall'ultimo layer si chiamano **logit**: sono numeri qualsiasi, positivi o negativi. Per trasformarli nel **valore finale** che ci interessa si applica una funzione che dipende dal compito:

| Compito | Uscite | Funzione sui logit | Valore finale |
|---|---|---|---|
| Regressione | 1 | nessuna (identità) | la stima di $y$ |
| Classificazione binaria | 1 | sigmoide | la probabilità della classe 1 |
| Classificazione a $k$ classi | $k$ | softmax | $k$ probabilità, una per classe, che sommano a 1 |

In PyTorch la rete restituisce i logit, e la funzione finale la applichiamo noi quando vogliamo leggere le uscite (nella Parte 4 vediamo perché). Creiamo tre reti con lo stesso corpo `[32, 16]` e guardiamo cosa rispondono ai primi 3 esempi, prima di qualsiasi addestramento.

In [ ]:
torch.manual_seed(0)
rete_reg = MLP(10, [32, 16], n_uscite=1)      # regressione
rete_bin = MLP(10, [32, 16], n_uscite=1)      # classificazione binaria
rete_cls = MLP(10, [32, 16], n_uscite=3)      # tre classi

esempi = X_train[:3]
with torch.no_grad():
    logit_reg, logit_bin, logit_cls = rete_reg(esempi), rete_bin(esempi), rete_cls(esempi)

print("REGRESSIONE, logit", tuple(logit_reg.shape), "= le stime di y:\n", logit_reg)
print("\nBINARIA, logit", tuple(logit_bin.shape), ":\n", logit_bin)
print("→ sigmoide → probabilità della classe 1:\n", torch.sigmoid(logit_bin))
print("\nTRE CLASSI, logit", tuple(logit_cls.shape), ":\n", logit_cls)
probabilita = torch.softmax(logit_cls, dim=1)                # dim=1: lungo le 3 classi di ogni riga
print("→ softmax → probabilità delle 3 classi:\n", probabilita)
print("Somma di ogni riga:", probabilita.sum(dim=1))

Per la binaria, un logit positivo dà una probabilità sopra 0.5 e la rete sceglie la classe 1. Per le tre classi la rete sceglie la classe con la probabilità più alta, che è anche quella con il logit più alto. Per ora le reti hanno pesi casuali: le stime della regressione sono vicine a 0 e le probabilità sono vicine a 1/2 e a 1/3, come chi tira a indovinare.

📌 Il **corpo** di una rete (i layer nascosti) non dipende dal compito. Cambiano solo **quante uscite** ha l'ultimo layer e **quale funzione** trasforma i logit nel valore finale.

---
# Parte 4 — Addestrare la rete

Per capire come si addestra una rete seguiamo la **rete di regressione** lungo tutta la catena, un anello alla volta:

$$X \;\xrightarrow{\text{forward}}\; \hat y \;\xrightarrow{\text{loss: confronto con } y}\; L \;\xrightarrow{\text{backward}}\; \text{gradienti} \;\xrightarrow{\text{aggiornamento}}\; \text{pesi nuovi}$$

La classificazione binaria e quella a tre classi sono negli approfondimenti ⏩ della 4.2 (le loss) e della 4.7 (l'addestramento).

### 4.1 Il forward pass: la rete fa una previsione
Nel **forward pass** gli esempi attraversano la rete dal primo all'ultimo layer, come nella 2.6, e all'uscita c'è la previsione $\hat y$. È solo un calcolo: la rete non sa se ha sbagliato. Passiamo 5 esempi di training alla rete di regressione, che ha ancora i pesi casuali.

In [ ]:
cinque, y_cinque = X_train[:5], yreg_train[:5]     # 5 esempi e il loro y vero
with torch.no_grad():
    stime = rete_reg(cinque)                       # forward pass: forma (5, 1)

pd.DataFrame({"y vero": y_cinque.ravel().numpy(), "y stimato": stime.ravel().numpy()}).round(2)

Le stime sono tutte vicine a 0, mentre i valori veri sono intorno a 14. Per migliorare, la rete ha bisogno di sapere **quanto** sta sbagliando, con un numero solo: è il lavoro della loss.

### 4.2 La loss: quanto sbaglia la rete, in un numero
La **loss** confronta i valori finali della rete con i target e restituisce **un numero**: piccolo se la rete ci prende, grande se sbaglia. È l'unica cosa che l'addestramento guarda: cercherà i parametri che la rendono piccola. Per la regressione la loss è l'MSE: vediamola su numeri piccoli, calcolata a mano e poi con PyTorch.

**Regressione: MSE (errore quadratico medio).** Per ogni esempio l'errore è $\hat y - y$; lo si eleva al quadrato (così gli errori negativi non compensano quelli positivi, e quelli grandi pesano di più) e si fa la media.

In [ ]:
y_vero = torch.tensor([10.0, 12.0, 9.0, 13.0])
y_stimato = torch.tensor([11.0, 11.5, 8.0, 15.0])

errori = y_stimato - y_vero
print("errori:     ", errori)
print("al quadrato:", errori ** 2)
print("MSE = media:", (errori ** 2).mean())
print("nn.MSELoss: ", nn.MSELoss()(y_stimato, y_vero))

Ogni compito ha la sua loss, calcolata a partire dal **valore finale** della Parte 3:

| Compito | Valore finale | Loss | In PyTorch, sui logit |
|---|---|---|---|
| Regressione | $\hat y$ | MSE | `nn.MSELoss()` |
| Binaria | $p$ = sigmoide(logit) | binary cross-entropy: $-[\,y \ln p + (1-y)\ln(1-p)\,]$ | `nn.BCEWithLogitsLoss()` |
| $k$ classi | $p$ = softmax(logit) | cross-entropy: $-\ln p_{\text{classe vera}}$ | `nn.CrossEntropyLoss()` |

Nota che `BCEWithLogitsLoss` e `CrossEntropyLoss` ricevono i **logit**, non le probabilità: la sigmoide e la softmax le fanno loro, insieme al logaritmo. È più stabile: se una probabilità venisse arrotondata a 0, $\ln 0$ darebbe $-\infty$ e l'addestramento si romperebbe. Per questo nel codice la rete restituisce i logit e la funzione finale la applichiamo solo quando vogliamo leggere le uscite. Le due loss della classificazione sono calcolate a mano nell'approfondimento qui sotto.

Ora la loss della nostra rete di regressione, ancora con i pesi casuali: sui 5 esempi della 4.1 e su tutto il training set.

In [ ]:
loss_reg = nn.MSELoss()

with torch.no_grad():
    print(f"MSE sui 5 esempi della 4.1:     {loss_reg(rete_reg(cinque), y_cinque).item():8.3f}")
    print(f"MSE su tutto il training set:   {loss_reg(rete_reg(X_train), yreg_train).item():8.3f}")

La rete parte con un MSE enorme, ben oltre 200: stima numeri vicini a 0, mentre $y$ vale in media 14.


### ⏩ Le loss della classificazione
**Classificazione binaria: BCE (*binary cross-entropy*).** Il valore finale è la probabilità $p$ della classe 1. Per ogni esempio

$$L = -\big[\, y \cdot \ln p + (1 - y) \cdot \ln(1 - p) \,\big]$$

Se $y = 1$ resta $-\ln p$: piccola se $p$ è vicina a 1, enorme se $p$ è vicina a 0. Se $y = 0$ resta $-\ln(1 - p)$. Partiamo da quattro logit, li passiamo nella sigmoide e calcoliamo la loss: due esempi indovinati con sicurezza ($p \approx 0.9$ per $y = 1$, $p \approx 0.1$ per $y = 0$) e due sbagliati con sicurezza.

In [ ]:
logit = torch.tensor([2.2, -2.2, -2.2, 2.2])
y_classe = torch.tensor([1.0, 1.0, 0.0, 0.0])

p = torch.sigmoid(logit)                                     # i valori finali: probabilità della classe 1
per_esempio = -(y_classe * torch.log(p) + (1 - y_classe) * torch.log(1 - p))
print("p:          ", p)
print("loss per esempio:", per_esempio)
print("BCE = media:", per_esempio.mean())
print("nn.BCEWithLogitsLoss sui logit:", nn.BCEWithLogitsLoss()(logit, y_classe))

Indovinare con sicurezza costa poco (circa $0.105$), sbagliare con sicurezza costa tanto (circa $2.30$).

**Classificazione a più classi: cross-entropy.** I valori finali sono le probabilità della softmax, e la loss è $-\ln p_{\text{classe vera}}$: conta solo la probabilità che la rete dà alla risposta giusta. Un esempio con 3 classi A, B, C, in cui la classe vera è A:

In [ ]:
logit = torch.tensor([[2.0, 1.0, 0.1]])          # un esempio, 3 classi: forma (1, 3)
classe_vera = torch.tensor([0])                  # la classe A (si conta da 0)

p = torch.softmax(logit, dim=1)                  # i valori finali: 3 probabilità
print("p:", p)
print("p della classe vera:", p[0, 0])
print("cross-entropy = -ln p:", -torch.log(p[0, 0]))
print("nn.CrossEntropyLoss sui logit:", nn.CrossEntropyLoss()(logit, classe_vera))

Infine le loss delle due reti di classificazione della Parte 3, ancora con i pesi casuali, su tutto il training set. Ogni loss riceve il **suo** target (1.4): una colonna di numeri con la virgola per la BCE, un vettore di interi per la cross-entropy.

In [ ]:
with torch.no_grad():
    print(f"Binaria (BCE):              {nn.BCEWithLogitsLoss()(rete_bin(X_train), ybin_train).item():.3f}   ln 2 = {np.log(2):.3f}")
    print(f"Tre classi (cross-entropy): {nn.CrossEntropyLoss()(rete_cls(X_train), ycls_train).item():.3f}   ln 3 = {np.log(3):.3f}")

Partono **alla pari con chi tira a indovinare**: con probabilità vicine a 1/2 (o a 1/3), la loss è circa $-\ln \frac{1}{2} = \ln 2$ (o $\ln 3$).

### 4.3 Il gradiente di un nodo, calcolato a mano
La loss dipende dalle stime, e le stime dipendono dai **parametri** della rete, cioè pesi e bias: quindi anche la loss dipende dai parametri. Il **gradiente** di un parametro dice quanto cambia la loss se quel parametro cambia di poco. Vediamo come si calcola su un esempio piccolissimo: il **nodo finale** di una rete di regressione, senza attivazione, con 2 ingressi $h_1, h_2$ (per esempio le uscite del layer nascosto precedente), 2 pesi e un bias:

$$\hat y = w_1 h_1 + w_2 h_2 + b \qquad\qquad L = (\hat y - y)^2$$

Con la **regola della catena**, il gradiente di ogni parametro è il prodotto di due pezzi: quanto cambia la loss se cambia la stima $\hat y$, per quanto cambia $\hat y$ se cambia quel parametro.

$$\frac{\partial L}{\partial w_1} = \underbrace{2\,(\hat y - y)}_{\text{quanto sbaglia}} \cdot \underbrace{h_1}_{\text{ingresso del peso}} \qquad\quad \frac{\partial L}{\partial w_2} = 2\,(\hat y - y) \cdot h_2 \qquad\quad \frac{\partial L}{\partial b} = 2\,(\hat y - y) \cdot 1$$

Il bias si comporta come un peso con l'ingresso fisso a 1. Usiamo questi numeri: $h = (1, 2)$, $w = (0.5, -1)$, $b = 0.5$, e il valore vero $y = 3$.

In [ ]:
h = torch.tensor([1.0, 2.0])         # i 2 ingressi del nodo
w = torch.tensor([0.5, -1.0])        # i suoi 2 pesi
b = torch.tensor(0.5)                # il suo bias
y_vero = torch.tensor(3.0)

y_stimato = w @ h + b                # forward: 0.5·1 + (-1)·2 + 0.5
loss = (y_stimato - y_vero) ** 2     # loss
dL_dy = 2 * (y_stimato - y_vero)     # quanto cambia la loss se cambia la stima
grad_w = dL_dy * h                   # errore × ingresso, per ogni peso
grad_b = dL_dy * 1                   # il bias ha ingresso 1

print(f"Forward:  y stimato = {y_stimato.item():.2f}   (vero: {y_vero.item():.2f})")
print(f"Loss:     L = ({y_stimato.item():.0f} - {y_vero.item():.0f})² = {loss.item():.0f}")
print(f"∂L/∂ŷ = 2 · ({y_stimato.item():.0f} - {y_vero.item():.0f}) = {dL_dy.item():.0f}")
print("Gradienti dei pesi ∂L/∂w = ∂L/∂ŷ · h:", grad_w)
print("Gradiente del bias ∂L/∂b = ∂L/∂ŷ · 1:", grad_b)

La stima è **troppo bassa** ($-1$ invece di $3$), quindi $\partial L / \partial \hat y$ è negativo, e lo sono anche tutti i gradienti: aumentando i parametri la loss scende. Il gradiente di $w_2$ è il doppio di quello di $w_1$ perché il suo ingresso è il doppio: un peso "ha più colpa" dell'errore se l'ingresso che moltiplica è grande.

Ora chiediamo gli stessi gradienti a PyTorch con `.backward()`: l'autograd applica la regola della catena per noi.

In [ ]:
w = torch.tensor([0.5, -1.0], requires_grad=True)
b = torch.tensor(0.5, requires_grad=True)

loss = (w @ h + b - y_vero) ** 2
loss.backward()                                   # l'autograd applica la regola della catena per noi
print("Gradienti di PyTorch:  ∂L/∂w =", w.grad, "  ∂L/∂b =", b.grad)

Il gradiente è la **pendenza** della loss. Lo vediamo disegnando la loss al variare di $w_1$, con tutti gli altri parametri fermi: nel punto attuale, $w_1 = 0.5$, la retta tangente ha pendenza $-8$, proprio il gradiente calcolato.

In [ ]:
#@title 📈 La loss al variare di w₁ e la sua pendenza (esegui la cella)
valori_w1 = np.linspace(-1, 8, 200)
curva = (valori_w1 * 1 + (-1) * 2 + 0.5 - 3) ** 2            # la loss con w₂ = -1 e b = 0.5 fissi
plt.figure(figsize=(7, 4))
plt.plot(valori_w1, curva, label="loss al variare di w₁")
vicini = np.linspace(-0.5, 1.5, 10)
plt.plot(vicini, 16 + (-8) * (vicini - 0.5), "r--", linewidth=2, label="tangente: pendenza = ∂L/∂w₁ = -8")
plt.plot(0.5, 16, "ko", label="valore attuale (w₁ = 0.5)")
plt.xlabel("w₁"); plt.ylabel("loss"); plt.ylim(-1, 35)
plt.legend(); plt.grid(alpha=0.3)
plt.show()

La pendenza è negativa: andando verso destra la loss scende, verso il minimo in $w_1 = 4.5$. Per far scendere la loss bisogna quindi **aumentare** $w_1$: è quello che fa l'aggiornamento dei parametri (4.4).

**Con un batch.** Con più esempi la loss è la **media** delle loss dei singoli esempi, e il gradiente di ogni parametro diventa la media di "errore × ingresso". Proviamo con 3 esempi: una tabella $H$ con un esempio per riga.

> 🐍 `errori.reshape(-1, 1) * H`: gli errori diventano una colonna (3, 1) e si moltiplicano per ogni colonna di $H$ (3, 2), cioè ogni errore per gli ingressi del suo esempio. È il *broadcasting* della 1.3.

In [ ]:
H = torch.tensor([[1.0, 2.0],             # esempio 1 (quello di prima)
                  [2.0, 0.0],             # esempio 2
                  [0.0, 1.0]])            # esempio 3: forma (3, 2)
y_veri = torch.tensor([3.0, 1.0, 2.0])

w = torch.tensor([0.5, -1.0], requires_grad=True)
b = torch.tensor(0.5, requires_grad=True)
stime = H @ w + b
errori = stime - y_veri
loss = (errori ** 2).mean()                            # MSE: la media sui 3 esempi
loss.backward()

with torch.no_grad():
    grad_w_mano = (2 * errori.reshape(-1, 1) * H).mean(dim=0)    # media di 2·errore × ingresso
    grad_b_mano = (2 * errori).mean()
print("stime:", stime.detach(), "  errori:", errori.detach(), f"  MSE: {loss.item():.2f}")
print("∂L/∂w → a mano:", grad_w_mano, "  PyTorch:", w.grad)
print("∂L/∂b → a mano:", grad_b_mano, "  PyTorch:", b.grad)

È quello che succede in ogni passo di addestramento: il gradiente di ogni parametro è calcolato su un **batch** di esempi, facendo la media dei contributi di tutti gli esempi.

### ⏩ Un passo indietro: il gradiente arriva al layer nascosto
I nodi nascosti non vedono direttamente la loss: il loro errore arriva **dal nodo finale**, passando per i pesi che li collegano. Allunghiamo la rete di prima di un layer: $h_1$ e $h_2$ diventano le uscite di **due nodi nascosti** con la ReLU, che ricevono un solo ingresso $x = 2$. Nello schema ogni nodo è diviso nei suoi due passaggi: il cerchio **Σ** fa la somma pesata dei rami che entrano (ognuno con il suo peso) più il bias, il blocco **ReLU** applica la funzione di attivazione. In blu i valori calcolati nel forward, in rosso i gradienti calcolati nel backward.

In [ ]:
#@title 🗺️ Lo schema della rete (esegui la cella)
from matplotlib.patches import Circle, FancyArrowPatch, Rectangle

blu, rosso = "#1f5fa8", "#c0392b"
fig, ax = plt.subplots(figsize=(21, 10.5))
ax.set_xlim(-1.5, 21.8); ax.set_ylim(-7.2, 7.2); ax.axis("off")

def scritta(x, y, testo, colore="k", **opzioni):
    opzioni.setdefault("fontsize", 11.5)
    ax.text(x, y, testo, ha="center", va="center", color=colore, zorder=6, linespacing=1.4,
            bbox=dict(facecolor="white", edgecolor="none", pad=1), **opzioni)

def freccia(x1, y1, x2, y2, colore="k", stile="-"):
    ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2), arrowstyle="-|>", mutation_scale=18, linewidth=1.5,
                                 color=colore, linestyle=stile, shrinkA=0, shrinkB=0, zorder=2))

def somma(x, y, colore):                         # il nodo che fa la somma pesata
    ax.add_patch(Circle((x, y), 0.5, facecolor=colore, edgecolor="k", linewidth=1.5, zorder=3))
    ax.text(x, y, "Σ", ha="center", va="center", fontsize=18, zorder=4)

def blocco_relu(x, y):                           # il blocco della ReLU, con il disegno della funzione
    ax.add_patch(Rectangle((x - 0.8, y - 0.5), 1.6, 1.0, facecolor="#e8f4e1", edgecolor="k", linewidth=1.5, zorder=3))
    ax.plot([x - 0.55, x, x + 0.45], [y - 0.25, y - 0.25, y + 0.28], color="k", linewidth=2, zorder=4)
    ax.text(x, y + (0.8 if y > 0 else -0.8), "ReLU", ha="center", va="center", fontsize=11.5,
            fontweight="bold", zorder=5)

def scatola(x, y, testo, colore="#eeeeee"):     # i bias e la loss
    ax.text(x, y, testo, ha="center", va="center", fontsize=12, zorder=5,
            bbox=dict(boxstyle="round,pad=0.4", facecolor=colore, edgecolor="k"))

# le parti della rete, in alto
for xx, nome in [(0, "ingresso"), (6.75, "layer nascosto: somma pesata Σ, poi ReLU"),
                 (15.5, "nodo finale: somma pesata Σ"), (19.9, "loss")]:
    scritta(xx, 5.55, nome, style="italic", fontsize=12.5)

# ingresso
ax.add_patch(Circle((0, 0), 0.5, facecolor="#dddddd", edgecolor="k", linewidth=1.5, zorder=3))
ax.text(0, 0, "x", ha="center", va="center", fontsize=16, fontweight="bold", zorder=4)
scritta(0, -0.95, "x = 2", blu)

# i due nodi nascosti: somma pesata Σ (peso v e bias c), poi ReLU
nodi = [("₁", 2.8, "0.5", "0", "0.5 · 2 + 0 = 1", "−4", "−8", "1", "0.5", "0.5", "−4", "−8"),
        ("₂", -2.8, "1.5", "−1", "1.5 · 2 − 1 = 2", "8", "16", "2", "−1", "(−1)", "8", "−16")]
for p, yy, v, c, z, dz, dv, h, w, w_tra_parentesi, dh, dw in nodi:
    s = 1 if yy > 0 else -1
    # ramo x → Σ, con il peso v
    somma(4.5, yy, "#cfe3f5")
    freccia(0.45, 0.25 * s, 4.05, yy - 0.25 * s)
    scritta(1.55, 1.85 * s, f"v{p} = {v}")
    scritta(3.1, 0.75 * s, f"∂L/∂v{p} = ∂L/∂z{p} · x\n= {dz} · 2 = {dv}", rosso)
    # bias c
    scatola(4.5, yy + 1.7 * s, f"bias c{p} = {c}")
    freccia(4.5, yy + 1.35 * s, 4.5, yy + 0.5 * s)
    scritta(7.4, yy + 1.7 * s, f"∂L/∂c{p} = ∂L/∂z{p} = {dz}", rosso)
    # la somma pesata
    scritta(4.5, yy - 1.1 * s, f"z{p} = v{p} · x + c{p}\n= {z}", blu)
    # Σ → ReLU
    freccia(5.0, yy, 8.2, yy)
    scritta(6.6, yy + 0.75 * s, f"∂L/∂z{p} = ∂L/∂h{p} · ReLU'(z{p})\n= {dh} · 1 = {dz}", rosso)
    blocco_relu(9, yy)
    # ReLU → nodo finale, con il peso w
    freccia(9.8, yy, 15.05, 0.3 * s)
    scritta(11.4, yy + 0.45 * s, f"h{p} = ReLU(z{p}) = {h}", blu)
    scritta(11.4, yy + 1.35 * s, f"∂L/∂h{p} = ∂L/∂ŷ · w{p}\n= −8 · {w_tra_parentesi} = {dh}", rosso)
    scritta(12.0, 1.55 * s, f"w{p} = {w}")
    scritta(12.0, 0.7 * s, f"∂L/∂w{p} = ∂L/∂ŷ · h{p}\n= −8 · {h} = {dw}", rosso)

# nodo finale
somma(15.5, 0, "#f9dcc4")
scatola(15.5, 2.0, "bias b = 0.5")
freccia(15.5, 1.65, 15.5, 0.5)
scritta(15.5, 2.85, "∂L/∂b = ∂L/∂ŷ · 1 = −8", rosso)
scritta(15.9, -1.35, "ŷ = w₁ · h₁ + w₂ · h₂ + b\n= 0.5 · 1 + (−1) · 2 + 0.5 = −1", blu)

# loss
freccia(16.0, 0, 18.75, 0)
scritta(17.4, 0.85, "∂L/∂ŷ = 2(ŷ − y)\n= 2 · (−1 − 3) = −8", rosso)
scatola(19.9, 0, "L = (ŷ − y)²", "#f4cccc")
scritta(19.9, -1.15, "y = 3\nL = (−1 − 3)² = 16", blu)

# le due direzioni
freccia(21.4, 6.4, -0.8, 6.4, colore=rosso, stile="--")
scritta(10.3, 6.8, "backward (gradienti, in rosso): dalla loss verso l'ingresso", rosso, fontsize=12.5)
freccia(-0.8, -6.4, 21.4, -6.4, colore=blu, stile="--")
scritta(10.3, -6.8, "forward (valori, in blu): dall'ingresso alla loss", blu, fontsize=12.5)
plt.show()

| Simbolo | Cos'è | Valore |
|---|---|---|
| $x$ | l'ingresso della rete | $2$ |
| $v_1, v_2$ | i pesi del layer nascosto, uno per nodo nascosto | $0.5,\ 1.5$ |
| $c_1, c_2$ | i bias del layer nascosto | $0,\ -1$ |
| $z_1, z_2$ | le somme pesate dei nodi nascosti: $z = v \cdot x + c$ | $1,\ 2$ |
| $h_1, h_2$ | le uscite dei nodi nascosti dopo la ReLU, che sono gli **ingressi** del nodo finale | $1,\ 2$ |
| $w_1, w_2, b$ | i pesi e il bias del nodo finale (gli stessi della 4.3) | $0.5,\ -1,\ 0.5$ |
| $\hat y$ | la stima della rete | $-1$ |
| $y$ | il valore vero | $3$ |
| $L$ | la loss | $16$ |

Gli $h$ sono proprio quelli della 4.3, quindi il nodo finale fa gli stessi conti di prima. Per i parametri del layer nascosto la regola della catena si allunga di un anello per ogni passaggio, andando **all'indietro** dalla loss:

$$\frac{\partial L}{\partial h_1} = \frac{\partial L}{\partial \hat y} \cdot w_1 \qquad \frac{\partial L}{\partial z_1} = \frac{\partial L}{\partial h_1} \cdot \text{ReLU}'(z_1) \qquad \frac{\partial L}{\partial v_1} = \frac{\partial L}{\partial z_1} \cdot x \qquad \frac{\partial L}{\partial c_1} = \frac{\partial L}{\partial z_1}$$

L'errore torna indietro **moltiplicato per il peso** che collega i due nodi; la ReLU lo lascia passare se $z > 0$ (derivata 1, come qui) o lo blocca se $z < 0$ (derivata 0); infine, come per il nodo finale, si moltiplica per l'ingresso del peso. È la **backpropagation**: si parte dalla loss e si torna indietro, un layer alla volta.

In [ ]:
x = torch.tensor(2.0)
v = torch.tensor([0.5, 1.5], requires_grad=True)     # pesi dei 2 nodi nascosti
c = torch.tensor([0.0, -1.0], requires_grad=True)    # bias dei 2 nodi nascosti
w = torch.tensor([0.5, -1.0], requires_grad=True)    # pesi del nodo finale
b = torch.tensor(0.5, requires_grad=True)            # bias del nodo finale

z = v * x + c                                        # layer nascosto: somme pesate...
h = torch.relu(z)                                    # ...e ReLU
y_stimato = w @ h + b                                # nodo finale
loss = (y_stimato - y_vero) ** 2
loss.backward()                                      # PyTorch calcola tutti i gradienti

with torch.no_grad():                                # e ora a mano, all'indietro
    dL_dy = 2 * (y_stimato - y_vero)                 # dal nodo finale: -8
    dL_dh = dL_dy * w                                # l'errore torna indietro passando per i pesi w
    dL_dz = dL_dh * (z > 0).float()                  # la ReLU lascia passare (1) o blocca (0)
print("z:", z.detach(), "  h:", h.detach(), f"  ŷ: {y_stimato.item():.1f}")
print("∂L/∂h:", dL_dh)
print("∂L/∂v → a mano:", dL_dz * x, "  PyTorch:", v.grad)
print("∂L/∂c → a mano:", dL_dz, "  PyTorch:", c.grad)

Gli stessi numeri. Con migliaia di parametri e tanti layer il conto è lo stesso, ripetuto: PyTorch lo fa con un solo `.backward()`.

### 4.4 Aggiornare i parametri: discesa del gradiente e learning rate
Il gradiente dice da che parte la loss **sale**. Per farla scendere si sposta ogni parametro di un passettino nella direzione **opposta**:

$$w \leftarrow w - \eta \cdot \frac{\partial L}{\partial w} \qquad\qquad b \leftarrow b - \eta \cdot \frac{\partial L}{\partial b}$$

dove $\eta$ (si legge *eta*) è il **learning rate**, la lunghezza del passo. Pesi e bias si aggiornano con la stessa regola. Facciamo un aggiornamento sul nodo della 4.3 con $\eta = 0.01$.

> 🐍 L'aggiornamento si fa dentro `with torch.no_grad():`, perché PyTorch non deve tenere traccia anche di questo calcolo. `w -= a` è un modo breve per scrivere `w = w - a`.

In [ ]:
h = torch.tensor([1.0, 2.0])                       # il nodo della 4.3: ingressi...
y_vero = torch.tensor(3.0)                         # ...e valore vero
w = torch.tensor([0.5, -1.0], requires_grad=True)
b = torch.tensor(0.5, requires_grad=True)
eta = 0.01

loss = (w @ h + b - y_vero) ** 2                   # forward + loss
loss.backward()                                    # gradienti
print("Prima:   w =", w.detach(), f"  b = {b.item():.2f}   ŷ = {(w @ h + b).item():.2f}   loss = {loss.item():.2f}")
print("Gradienti: ∂L/∂w =", w.grad, f"  ∂L/∂b = {b.grad.item():.0f}")

with torch.no_grad():                              # aggiornamento
    w -= eta * w.grad                              # w ← w − η · ∂L/∂w
    b -= eta * b.grad                              # b ← b − η · ∂L/∂b
    y_stimato = w @ h + b
print("Dopo:    w =", w.detach(), f"  b = {b.item():.2f}   ŷ = {y_stimato.item():.2f}   loss = {((y_stimato - y_vero) ** 2).item():.2f}")

Tutti i gradienti erano negativi, quindi tutti i parametri sono **aumentati**; $w_2$, che aveva il gradiente più grande, si è spostato il doppio di $w_1$. La stima è passata da $-1$ a $-0.52$, più vicina al valore vero $3$, e la loss è scesa da $16$ a $12.39$.

Un passo solo non basta. La **discesa del gradiente** ripete tante volte gli stessi tre passaggi: forward + loss, gradienti, aggiornamento. A ogni giro i gradienti vanno ricalcolati, perché i parametri sono cambiati. Qui, per vedere il meccanismo, ripetiamo i passi sempre sullo **stesso esempio**; nell'addestramento vero ogni passo usa un **batch diverso** di esempi (4.6).

> 🐍 `w.grad.zero_()` **azzera** il gradiente dopo l'aggiornamento: PyTorch **somma** i nuovi gradienti a quelli che ci sono già, e senza azzerarli al giro dopo avremmo la somma dei gradienti di tutti i giri.

In [ ]:
w = torch.tensor([0.5, -1.0], requires_grad=True)  # ripartiamo dagli stessi valori
b = torch.tensor(0.5, requires_grad=True)
eta = 0.01

for passo in range(1, 31):
    loss = (w @ h + b - y_vero) ** 2               # 1. forward + loss
    loss.backward()                                # 2. gradienti
    with torch.no_grad():                          # 3. aggiornamento
        w -= eta * w.grad
        b -= eta * b.grad
    w.grad.zero_()                                 # azzera i gradienti per il giro dopo
    b.grad.zero_()
    if passo in (1, 2, 3, 5, 10, 20, 30):
        print(f"passo {passo:2d}:  loss {loss.item():8.4f}   →   w = {w.detach().numpy().round(3)}, b = {b.item():.3f}")

print(f"\nDopo 30 passi: ŷ = {(w @ h + b).item():.3f}   (valore vero 3)")

La loss scende a ogni passo, e la stima arriva quasi a $3$. I passi si accorciano da soli: più ci si avvicina, più l'errore e quindi i gradienti sono piccoli.

### Il learning rate
Quanto deve essere lungo il passo? Per vederlo usiamo una loss ancora più semplice, che dipende da **un solo parametro** $w$ e ha il minimo in $w = 2$:

$$L(w) = (w - 2)^2 + 0.5 \qquad\qquad L'(w) = 2\,(w - 2)$$

Partendo da $w = 0$, facciamo 12 passi $w \leftarrow w - \eta \cdot L'(w)$ con quattro learning rate diversi, e disegniamo i punti visitati sulla curva della loss.

In [ ]:
#@title 📈 Discesa del gradiente con quattro learning rate (esegui la cella)
def L(w):
    return (w - 2) ** 2 + 0.5

curva_w = np.linspace(-1.5, 5.5, 200)
fig, assi = plt.subplots(1, 4, figsize=(18, 3.8), sharey=True)
for ax, eta in zip(assi, [0.02, 0.1, 0.95, 1.05]):
    punti = [0.0]
    for passo in range(12):
        punti.append(punti[-1] - eta * 2 * (punti[-1] - 2))      # w ← w − η · L'(w)
    punti = np.array(punti)
    ax.plot(curva_w, L(curva_w), color="gray")
    ax.plot(punti, L(punti), "o-", color="C3", markersize=4, linewidth=1)
    ax.plot(punti[0], L(punti[0]), "s", color="k", markersize=7, label="partenza")
    ax.set_title(f"η = {eta}: dopo 12 passi w = {punti[-1]:.2f}")
    ax.set_xlabel("w"); ax.set_ylim(0, 14); ax.set_xlim(-1.5, 5.5); ax.grid(alpha=0.3)
assi[0].set_ylabel("L(w)"); assi[0].legend()
plt.tight_layout()
plt.show()

> 🤔 Cosa succede con i quattro learning rate?

<details>
<summary><b>💡 Risposta</b> (clicca per aprire, ma prima prova a rispondere!)</summary>

- **$\eta = 0.02$, troppo piccolo**: i passi sono minuscoli e dopo 12 passi $w$ è ancora lontano da 2.
- **$\eta = 0.1$, adatto**: scende in fretta e si avvicina al minimo con passi sempre più corti.
- **$\eta = 0.95$, grande**: ogni passo scavalca il minimo e finisce dall'altra parte. Si avvicina lo stesso, ma rimbalzando, e dopo 12 passi è più lontano che con $\eta = 0.1$.
- **$\eta = 1.05$, troppo grande**: ogni passo scavalca il minimo e finisce **più lontano** di prima: la discesa **diverge**.

</details>

📌 Il **learning rate** è l'iperparametro più importante dell'addestramento. Non esiste un valore giusto per tutti i problemi: in pratica si provano valori su scala logaritmica (0.1, 0.01, 0.001, …) e si guarda la curva della loss.

### 4.5 Il passo completo sulla rete
Ora mettiamo insieme tutti gli anelli, sulla rete di regressione. L'addestramento ripete sempre gli stessi quattro passi:

1. **azzerare i gradienti** rimasti dal passo precedente;
2. **forward + loss**: un **batch** di esempi attraversa la rete (4.1) e la loss confronta le stime con i target (4.2): è la **media** degli errori sugli esempi del batch. Durante il forward PyTorch si ricorda i valori intermedi di ogni layer, perché al passo dopo gli servono;
3. **backward**: la **backpropagation** parte dalla loss e torna indietro layer per layer con la regola della catena, calcolando il gradiente della loss rispetto a **ogni** parametro, pesi e bias, come nella 4.3 ma per tutti i parametri insieme;
4. **aggiornamento**: l'ottimizzatore sposta ogni parametro contro il suo gradiente, come nella 4.4.

Facciamo **un solo** passo con un batch di 32 esempi e l'ottimizzatore più semplice, `SGD`, che è proprio la discesa del gradiente della 4.4, con $\eta = 0.01$:

$$W \leftarrow W - \eta \cdot \frac{\partial L}{\partial W}$$

> 🐍 `.detach().clone()` fa una **copia** indipendente di un tensore: senza, `pesi_prima` cambierebbe insieme alla rete.

In [ ]:
xb, yb = X_train[:32], yreg_train[:32]                     # un batch di 32 esempi
ottimizzatore = torch.optim.SGD(rete_reg.parameters(), lr=0.01)
pesi_prima = rete_reg.strati[0].weight.detach().clone()    # copia dei pesi del primo layer

ottimizzatore.zero_grad()                                  # 1. azzera i gradienti
stime = rete_reg(xb)                                       # 2. forward...
loss = loss_reg(stime, yb)                                 #    ...e loss
print("2. Forward: stime dei primi 3 esempi", stime[:3].detach().ravel(), "  valori veri", yb[:3].ravel())
print(f"   Loss sul batch: {loss.item():.2f}")

loss.backward()                                            # 3. backward: riempie .grad di ogni peso
gradiente = rete_reg.strati[0].weight.grad
print("3. Backward: gradiente dei pesi del primo layer, forma", tuple(gradiente.shape),
      "= forma dei pesi", tuple(pesi_prima.shape))

ottimizzatore.step()                                       # 4. aggiornamento
pesi_dopo = rete_reg.strati[0].weight.detach()
print("4. Aggiornamento: pesi dopo = pesi prima - 0.01 · gradiente?",
      torch.allclose(pesi_dopo, pesi_prima - 0.01 * gradiente))

with torch.no_grad():
    print(f"\nLoss sul batch: prima {loss.item():.2f}, dopo {loss_reg(rete_reg(xb), yb).item():.2f}")

Un passo sposta poco. Ripetiamo i quattro passi 30 volte, sempre sullo stesso batch, e guardiamo la loss:

In [ ]:
storia_passi = []
for passo in range(30):
    ottimizzatore.zero_grad()                  # 1.
    loss = loss_reg(rete_reg(xb), yb)          # 2.
    loss.backward()                            # 3.
    ottimizzatore.step()                       # 4.
    storia_passi.append(loss.item())

plt.figure(figsize=(7, 3.5))
plt.plot(range(1, 31), storia_passi, "o-", markersize=4)
plt.xlabel("passo"); plt.ylabel("MSE sul batch"); plt.title("30 passi sullo stesso batch di 32 esempi")
plt.grid(alpha=0.3)
plt.show()
print(f"Loss al primo passo: {storia_passi[0]:.2f}   all'ultimo: {storia_passi[-1]:.2f}")

La loss scende passo dopo passo: i quattro passi, ripetuti, fanno davvero imparare la rete. Attenzione però: abbiamo usato sempre **gli stessi 32 esempi**, quindi la rete sta imparando quel batch, non il problema. Nella 4.6 facciamo la stessa cosa su tutto il training set, un mini-batch diverso a ogni passo.

### 4.6 Mini-batch ed epoche
Ogni volta che si fa il backward si aggiorna anche la rete: ma **dopo quanti esempi**? Ci sono tre possibilità:

| Backward e aggiornamento dopo… | Esempi per passo | Pro e contro |
|---|---|---|
| ogni esempio (*SGD puro*) | 1 | tanti aggiornamenti, ma ognuno guarda un esempio solo: il percorso è molto rumoroso, e la GPU calcola un esempio alla volta |
| ogni **mini-batch** | di solito 32–256 | il compromesso standard: il gradiente è la media su un gruppetto di esempi, abbastanza stabile, e la GPU calcola il batch in parallelo |
| tutto il training set (*full batch*) | tutti | il gradiente più preciso, ma un solo aggiornamento per ogni giro sui dati, e con tanti dati non entra in memoria |

Noi usiamo mini-batch da 32 esempi presi **a caso**. Un'**epoca** è un giro completo sul training set: con 1200 esempi e batch da 32 sono 38 mini-batch (l'ultimo più piccolo), cioè **38 aggiornamenti per epoca**. `DataLoader` prepara i mini-batch e li rimescola a ogni epoca; `TensorDataset` mette insieme feature e target.

La funzione `addestra` riceve la rete, la loss e i dati. Per ogni epoca fa i quattro passi su ogni mini-batch, poi misura la loss su tutto il training e su tutto il **validation set**. Il validation set è fatto di esempi che la rete non usa mai per calcolare i gradienti: se la loss scende anche su di loro, la rete sta imparando qualcosa che vale anche per esempi nuovi; se scende solo sul training, sta imparando a memoria. Lo useremo nella Parte 5 per scegliere la rete e per capire quando fermarsi.

Come ottimizzatore usiamo **Adam** invece dello `SGD` della 4.5. SGD sposta tutti i parametri con lo stesso learning rate; Adam adatta il passo di ogni parametro in base ai suoi gradienti precedenti, e per questo funziona bene quasi sempre con il suo valore di default, `lr=0.001`, senza dover cercare il learning rate giusto.

Due cose nuove: `rete.train()` e `rete.eval()` dicono alla rete se la stiamo **addestrando** o **usando** (cambia qualcosa solo per il dropout, Parte 5), e `with torch.no_grad():` evita di calcolare gradienti quando misuriamo soltanto.

**Completa le tre righe mancanti** nel ciclo sui mini-batch: sono i passi 1, 3 e 4 della 4.5.

⚠️ Le celle successive usano questa funzione: se non riesci, **copia la soluzione** prima di andare avanti.

In [ ]:
def addestra(rete, funzione_loss, X_tr, y_tr, X_va, y_va, epoche=100, lr=0.001):
    """Addestra la rete con Adam e mini-batch da 32; restituisce una tabella con la loss per epoca."""
    torch.manual_seed(0)                                   # mini-batch rimescolati sempre allo stesso modo
    loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=32, shuffle=True)
    ottimizzatore = torch.optim.Adam(rete.parameters(), lr=lr)
    storia = []
    for epoca in range(epoche):
        rete.train()                                       # modalità addestramento
        for xb, yb in loader:                              # un mini-batch alla volta
            # ✏️ 1. azzera i gradienti

            loss = funzione_loss(rete(xb), yb)             # 2. forward + loss (questa c'è già)
            # ✏️ 3. backward

            # ✏️ 4. aggiornamento dei pesi

        rete.eval()                                        # modalità valutazione
        with torch.no_grad():
            loss_tr = funzione_loss(rete(X_tr), y_tr).item()
            loss_va = funzione_loss(rete(X_va), y_va).item()
        storia.append({"epoca": epoca + 1, "loss_train": loss_tr, "loss_val": loss_va})
    return pd.DataFrame(storia)

<details>
<summary><b>💡 Soluzione</b> (clicca per aprire, ma prima prova da solo!)</summary>

```python
        for xb, yb in loader:
            ottimizzatore.zero_grad()                      # 1. azzera i gradienti
            loss = funzione_loss(rete(xb), yb)             # 2. forward + loss
            loss.backward()                                # 3. backward
            ottimizzatore.step()                           # 4. aggiornamento dei pesi
```

Senza queste righe la rete calcola la loss ma non cambia mai i pesi: le curve restano piatte, ferme all'MSE di partenza.

</details>

Ora addestriamo la rete di regressione **da capo**, con pesi casuali nuovi, per 100 epoche, e la confrontiamo con una regressione lineare di scikit-learn, che può tracciare solo un iperpiano.

In [ ]:
from sklearn.linear_model import LinearRegression

torch.manual_seed(0)
rete_reg = MLP(10, [32, 16], n_uscite=1)
storia_reg = addestra(rete_reg, loss_reg, X_train, yreg_train, X_val, yreg_val, epoche=100)

lineare = LinearRegression().fit(X_train.numpy(), yreg_train.numpy())
with torch.no_grad():
    stima_rete = rete_reg(X_val).numpy().ravel()
stima_lineare = lineare.predict(X_val.numpy()).ravel()
veri = yreg_val.numpy().ravel()

print(f"MSE di validazione → rete: {np.mean((stima_rete - veri) ** 2):.2f}   regressione lineare: {np.mean((stima_lineare - veri) ** 2):.2f}")
disegna_curve({"[32, 16]": storia_reg}, "Regressione (MSE)", max_loss=30)
confronta_previsioni(veri, [stima_lineare, stima_rete], ["Regressione lineare", "Rete [32, 16]"])

Le due curve scendono insieme e restano vicine: la rete migliora anche sugli esempi di validazione, che non usa per calcolare i gradienti. L'MSE di validazione della rete è molto più basso di quello della regressione lineare: i dati di `make_friedman1` contengono termini non lineari (un seno, un quadrato), che un iperpiano non può seguire. Lo vedi anche nei grafici delle previsioni: i punti della rete stanno molto più vicini alla diagonale.

### 4.7 ⏩ Classificazione binaria e a tre classi, con la stessa funzione
La funzione `addestra` va bene anche per la classificazione. Per la **binaria** cambiano solo la loss e il target: l'ultimo layer ha comunque 1 uscita. L'accuracy si calcola scegliendo la classe 1 quando il logit è positivo, cioè quando la probabilità è sopra 0.5.

In [ ]:
torch.manual_seed(0)
rete_bin = MLP(10, [32, 16], n_uscite=1)
storia_bin = addestra(rete_bin, nn.BCEWithLogitsLoss(), X_train, ybin_train, X_val, ybin_val, epoche=50)

with torch.no_grad():
    accuracy_bin = ((rete_bin(X_val) >= 0).float() == ybin_val).float().mean().item()   # logit ≥ 0 ⇔ probabilità ≥ 0.5
print(storia_bin.round(3).iloc[[0, 9, 29, 49]])
print(f"\nAccuracy sulla validazione: {accuracy_bin:.3f}")
disegna_curve({"[32, 16]": storia_bin}, "Classificazione binaria (BCE)")

Con la stessa funzione `addestra`, la BCE scende come scendeva l'MSE, e la rete classifica correttamente la grande maggioranza degli esempi di validazione. Le classi sono bilanciate (la soglia è la mediana), quindi indovinare a caso darebbe un'accuracy di circa 0.5.

### ✏️ Esercizio: la rete per tre classi
Ora tocca a te: addestra una rete per il compito a **tre classi** (`ycls_train`, `ycls_val`). Rispetto alla rete binaria devi cambiare due cose: il numero di uscite e la loss. Sostituisci i due `None`, poi esegui la cella: calcola anche l'accuracy, scegliendo per ogni esempio la classe con il logit più alto (`argmax`).

In [ ]:
torch.manual_seed(0)
rete_cls = None            # ✏️ la rete per 3 classi
funzione_loss_cls = None   # ✏️ la loss per 3 classi

if rete_cls is not None and funzione_loss_cls is not None:
    storia_cls = addestra(rete_cls, funzione_loss_cls, X_train, ycls_train, X_val, ycls_val, epoche=50)
    with torch.no_grad():
        scelta = rete_cls(X_val).argmax(dim=1)                 # per ogni riga, la posizione del logit più alto
    print(f"Accuracy sulla validazione: {(scelta == ycls_val).float().mean().item():.3f}")
    disegna_curve({"[32, 16]": storia_cls}, "Tre classi (cross-entropy)")
else:
    print("✏️ Completa le due righe con None")

<details>
<summary><b>💡 Soluzione</b> (clicca per aprire, ma prima prova da solo!)</summary>

```python
rete_cls = MLP(10, [32, 16], n_uscite=3)
funzione_loss_cls = nn.CrossEntropyLoss()
```

Le uscite sono 3, una per classe, e la loss è la cross-entropy, che riceve i logit. Il target `ycls_train` contiene numeri interi 0, 1, 2 (tipo `long`, come abbiamo preparato nella 1.4) e non va trasformato. L'accuracy di solito è più bassa di quella binaria: con tre classi ci sono due confini da imparare invece di uno, e gli esempi vicini ai confini sono di più. Va comunque confrontata con 0.33, l'accuracy di chi tira a indovinare.

</details>

📌 Tre compiti, **stesso corpo**, stessa funzione di addestramento. Sono cambiati solo il numero di uscite dell'ultimo layer, la loss e il tipo di target.

---
# Parte 5 — Quanto grande deve essere la rete?

### 5.1 Pochi dati: underfitting e overfitting
Con 1200 esempi di training la rete `[32, 16]` funziona bene. Vediamo cosa succede con **solo 150 esempi**, sulla regressione, confrontando tre reti di dimensioni molto diverse:
- `[4]`: un solo layer nascosto con 4 neuroni (49 parametri);
- `[32, 16]`: la rete di prima (897 parametri);
- `[256, 256, 256]`: tre layer da 256 neuroni (134 657 parametri, quasi 900 per ogni esempio di training).

Le addestriamo per 300 epoche (con 150 esempi, un'epoca sono solo 5 mini-batch).

In [ ]:
X_poco, yreg_poco = X_train[:150], yreg_train[:150]       # solo i primi 150 esempi di training

storie = {}
for neuroni in ([4], [32, 16], [256, 256, 256]):
    torch.manual_seed(0)
    r = MLP(10, neuroni, n_uscite=1)
    storie[str(neuroni)] = addestra(r, loss_reg, X_poco, yreg_poco, X_val, yreg_val, epoche=300)

for nome, storia in storie.items():
    ultima = storia.iloc[-1]
    print(f"{nome:16s} MSE training {ultima['loss_train']:6.2f}   MSE validazione {ultima['loss_val']:6.2f}"
          f"   (minimo in validazione {storia['loss_val'].min():.2f} all'epoca {storia['loss_val'].idxmin() + 1})")
disegna_curve(storie, "Regressione con 150 esempi di training", max_loss=30)

> 🤔 Quale rete va in **underfitting** e quale in **overfitting**? Come lo riconosci dalle curve?

<details>
<summary><b>💡 Risposta</b> (clicca per aprire, ma prima prova a rispondere!)</summary>

- **`[4]` va in underfitting**: la loss di training e quella di validazione restano alte e vicine tra loro. La rete è troppo piccola per seguire i dati, anche quelli su cui si addestra.
- **`[256, 256, 256]` va in overfitting**: la loss di training scende quasi a zero, mentre quella di validazione arriva al suo minimo presto e poi risale. Con quasi 900 parametri per esempio la rete riesce a imparare a memoria i 150 esempi, rumore compreso.
- **`[32, 16]`** sta nel mezzo.

Si riconoscono dalla **distanza tra le due curve**: underfitting = entrambe alte; overfitting = training molto più basso della validazione, con la validazione che peggiora mentre il training migliora.

</details>

### 5.2 Il dropout
Il **dropout** spegne a caso una parte dei neuroni nascosti a ogni passo di addestramento (qui il 30%), ogni volta diversi: la rete non può affidarsi a pochi neuroni e deve distribuire quello che impara. In valutazione (`rete.eval()`) i neuroni sono tutti accesi. Nella nostra `MLP` basta il parametro `dropout`.

In [ ]:
torch.manual_seed(0)
rete_dropout = MLP(10, [256, 256, 256], n_uscite=1, dropout=0.3)
print(rete_dropout.strati[:4])                             # i primi layer: Linear, ReLU, Dropout, Linear

storia_dropout = addestra(rete_dropout, loss_reg, X_poco, yreg_poco, X_val, yreg_val, epoche=300)
ultima = storia_dropout.iloc[-1]
print(f"\nCon dropout: MSE training {ultima['loss_train']:.2f}, validazione {ultima['loss_val']:.2f}")
disegna_curve({"[256, 256, 256]": storie["[256, 256, 256]"], "[256, 256, 256] + dropout 0.3": storia_dropout},
              "Dropout con 150 esempi di training", max_loss=30)

Con il dropout la loss di training non arriva più quasi a zero: con il 30% dei neuroni spenti a ogni passo, la rete fa più fatica a imparare a memoria i 150 esempi. Confronta le due curve di validazione e la distanza tra training e validazione: è questo che il dropout prova a ridurre. Non fa miracoli: con così pochi dati resta un limite a quello che la rete può imparare.

### 5.3 L'early stopping
L'**early stopping** guarda la loss di validazione a ogni epoca: tiene una copia dei pesi dell'epoca **migliore** e ferma l'addestramento quando non migliora per un certo numero di epoche di fila, la *pazienza*. Alla fine la rete torna ai pesi migliori.

La funzione qui sotto è `addestra` della 4.6 con l'aggiunta delle righe segnate con `# ES`.

> 🐍 `rete.state_dict()` è un dizionario con tutti i pesi della rete; `copy.deepcopy` ne fa una copia indipendente (altrimenti la "copia" cambierebbe insieme alla rete); `rete.load_state_dict(...)` rimette i pesi salvati. `float("inf")` è l'infinito: qualsiasi loss sarà più piccola.

In [ ]:
def addestra_con_early_stopping(rete, funzione_loss, X_tr, y_tr, X_va, y_va, epoche=300, lr=0.001, pazienza=20):
    torch.manual_seed(0)
    loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=32, shuffle=True)
    ottimizzatore = torch.optim.Adam(rete.parameters(), lr=lr)
    storia = []
    migliore, pesi_migliori, attesa = float("inf"), None, 0    # ES: la loss migliore finora e i suoi pesi
    for epoca in range(epoche):
        rete.train()
        for xb, yb in loader:
            ottimizzatore.zero_grad()
            loss = funzione_loss(rete(xb), yb)
            loss.backward()
            ottimizzatore.step()
        rete.eval()
        with torch.no_grad():
            loss_tr = funzione_loss(rete(X_tr), y_tr).item()
            loss_va = funzione_loss(rete(X_va), y_va).item()
        storia.append({"epoca": epoca + 1, "loss_train": loss_tr, "loss_val": loss_va})
        if loss_va < migliore:                                  # ES: migliorata → salvo una copia dei pesi
            migliore, pesi_migliori, attesa = loss_va, copy.deepcopy(rete.state_dict()), 0
        else:                                                   # ES: non migliorata → aspetto ancora
            attesa += 1
            if attesa == pazienza:                              # ES: troppe epoche senza migliorare: stop
                break
    rete.load_state_dict(pesi_migliori)                         # ES: torno ai pesi migliori
    return pd.DataFrame(storia)

torch.manual_seed(0)
rete_es = MLP(10, [256, 256, 256], n_uscite=1)
storia_es = addestra_con_early_stopping(rete_es, loss_reg, X_poco, yreg_poco, X_val, yreg_val, pazienza=20)

with torch.no_grad():
    mse_val_es = loss_reg(rete_es(X_val), yreg_val).item()
migliore = storia_es["loss_val"].idxmin() + 1
print(f"Fermata all'epoca {len(storia_es)} su 300; pesi dell'epoca {migliore} → MSE di validazione {mse_val_es:.2f}")
disegna_curve({"[256, 256, 256] + early stopping": storia_es}, "Early stopping con pazienza 20", max_loss=30)

L'addestramento si ferma molto prima delle 300 epoche, e la rete torna ai pesi dell'epoca con la loss di validazione più bassa. Il risultato è quello del punto migliore della curva di validazione della 5.1, invece di quello dell'ultima epoca, quando la rete aveva già iniziato a imparare a memoria. In più si risparmia il tempo delle epoche inutili.

### 5.4 Valutazione finale sul test set
Come modello finale teniamo la rete `[32, 16]` della 4.6, addestrata su tutto il training set, e la confrontiamo con la regressione lineare sul test set.

In [ ]:
with torch.no_grad():
    mse_test_rete = loss_reg(rete_reg(X_test), yreg_test).item()
mse_test_lineare = np.mean((lineare.predict(X_test.numpy()) - yreg_test.numpy()) ** 2)
print(f"MSE sul test → rete [32, 16]: {mse_test_rete:.2f}   regressione lineare: {mse_test_lineare:.2f}")

Sul test set, che non è stato usato né per i gradienti né per scegliere la rete, la rete resta nettamente migliore della regressione lineare, e l'MSE è vicino a quello di validazione della 4.6: il risultato visto in validazione vale anche su esempi del tutto nuovi.

### 5.5 ⏩ Gradienti che svaniscono
Durante il backward, il gradiente di un peso dei primi layer è un **prodotto** di tante derivate, una per ogni layer che c'è dopo. La derivata della sigmoide non supera mai $0.25$ (lo vedi nel grafico della 2.3): in una rete profonda con la sigmoide il prodotto diventa minuscolo e i primi layer quasi non imparano. La derivata della ReLU invece vale 1 per tutti i valori positivi.

Costruiamo due reti profonde, 10 layer nascosti da 32 neuroni, una con la sigmoide e una con la ReLU. Facciamo un solo backward sulla regressione e misuriamo **quanto è grande il gradiente** dei pesi di ogni layer (la sua *norma*).

In [ ]:
def norme_dei_gradienti(attivazione, profondita=10):
    torch.manual_seed(0)
    strati, n_prima = [], 10
    for _ in range(profondita):
        strati += [nn.Linear(n_prima, 32), attivazione()]
        n_prima = 32
    strati.append(nn.Linear(32, 1))
    rete_profonda = nn.Sequential(*strati)
    loss_reg(rete_profonda(X_train), yreg_train).backward()
    return [s.weight.grad.norm().item() for s in rete_profonda if isinstance(s, nn.Linear)]

norme_sigmoide = norme_dei_gradienti(nn.Sigmoid)
norme_relu = norme_dei_gradienti(nn.ReLU)

posizioni = np.arange(1, len(norme_relu) + 1)
plt.figure(figsize=(9, 3.8))
plt.bar(posizioni - 0.2, norme_relu, width=0.4, label="ReLU")
plt.bar(posizioni + 0.2, norme_sigmoide, width=0.4, label="sigmoide")
plt.yscale("log")                                         # scala logaritmica: ogni tacca è ×10
plt.xlabel("layer (1 = il più vicino all'ingresso)"); plt.ylabel("norma del gradiente")
plt.title("Quanto è grande il gradiente in ogni layer")
plt.legend(); plt.grid(alpha=0.3)
plt.show()
print(f"Primo layer → ReLU: {norme_relu[0]:.2e}   sigmoide: {norme_sigmoide[0]:.2e}")

Nella rete con la ReLU il gradiente ha una grandezza simile in tutti i layer. Nella rete con la sigmoide diventa sempre più piccolo andando verso l'ingresso: nel primo layer è più piccolo di vari ordini di grandezza (ogni tacca dell'asse è un fattore 10). Con gradienti così piccoli l'aggiornamento $w - \eta \cdot \text{gradiente}$ quasi non sposta i pesi dei primi layer. È uno dei motivi per cui nei layer nascosti si usa la ReLU.

---
# Riepilogo

| Concetto | Dove l'abbiamo visto | In PyTorch |
|---|---|---|
| Standardizzazione (solo sul training) | 1.2 | `StandardScaler().fit(train)` |
| Tensori, tipi e forme | 1.3 | `torch.tensor`, `dtype`, `.shape`, `.reshape(-1, 1)` |
| Percettrone: somma pesata + bias + attivazione | 2.1 | `w @ x + b`, `torch.relu`, `torch.sigmoid` |
| Layer denso: $X W^T + b$ | 2.2 | `nn.Linear`, `.weight`, `.bias` |
| Funzioni di attivazione e loro derivate | 2.3 | `nn.Sigmoid`, `nn.Tanh`, `nn.ReLU`, `nn.LeakyReLU`, `nn.GELU`, `torch.autograd.grad` |
| Layer in fila, parametri | 2.4 | `nn.Sequential`, `.parameters()`, `.numel()` |
| Il percettrone multistrato (MLP) come classe | 2.5 | `nn.Module`, `__init__`, `forward` |
| Rappresentazioni interne | 2.6 | un `for` sui layer, `torch.no_grad()` |
| Ultimo layer: logit e valore finale per ogni compito | 3 | `torch.sigmoid`, `torch.softmax` |
| Forward pass | 4.1 | `rete(X)` |
| La loss della regressione: MSE | 4.2 | `nn.MSELoss` |
| Le loss della classificazione: binary cross-entropy, cross-entropy | ⏩ 4.2 | `nn.BCEWithLogitsLoss`, `nn.CrossEntropyLoss` (sui logit) |
| Il gradiente di un nodo: regola della catena, errore × ingresso, batch | 4.3 | `loss.backward()`, `.grad` |
| Backpropagation verso il layer nascosto | ⏩ 4.3 | `loss.backward()` |
| Discesa del gradiente e learning rate | 4.4 | `requires_grad`, `.backward()`, `.grad`, `torch.no_grad()` |
| I quattro passi dell'addestramento | 4.5 | `zero_grad()`, `loss.backward()`, `step()`, `torch.optim.SGD` |
| Mini-batch ed epoche, il ciclo di addestramento | 4.6 | `TensorDataset`, `DataLoader`, `torch.optim.Adam` |
| Stessa funzione per la classificazione | ⏩ 4.7 | `argmax` |
| Underfitting e overfitting | 5.1 | curve di training e validazione |
| Dropout ed early stopping | 5.2, 5.3 | `nn.Dropout`, `train()` / `eval()`, `state_dict()` |
| Gradienti che svaniscono | ⏩ 5.5 | `.grad.norm()` |